Imports + Config + Seed

In [1]:
import math
import random
from dataclasses import dataclass
from typing import Dict, Tuple, Optional, List

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class CFG:
    # data
    ml100k_path: str = "ml100k_ratings.csv"  
    sep: str = ";"

    # time binning
    snapshot_gran: str = "d"  # 'h' (hour) or 'd' (day). Для ML-100K 'd' обычно адекватнее.
    window_sids: int = 0      # 0 = весь префикс, иначе sliding window в количестве sid

    # model
    node_dim: int = 128       # размер node_emb (вход GCN)
    embed_dim: int = 128      # размер z (выход энкодера)
    n_layers: int = 2
    dropout: float = 0.2

    # training
    lr: float = 1e-3
    weight_decay: float = 0.0
    epochs: int = 50

    # split
    train_ratio: float = 0.70
    val_ratio: float = 0.15
    test_ratio: float = 0.15

    # eval
    k: int = 20

    # system
    seed: int = 1337
    device: str =  "cpu"


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


cfg = CFG()
seed_everything(cfg.seed)
device = torch.device(cfg.device)
device

device(type='cpu')

Load MovieLens-100K → df(from,to,timestamp)

In [2]:
def load_ml100k_as_events(path: str, sep: str = "\t") -> pd.DataFrame:
    df = pd.read_csv(
        path,
        sep=sep,
        header=0,
        names=["user_id", "item_id", "timestamp", "rating"],
        engine="python",
    )
    df = df.dropna(subset=["user_id", "item_id", "timestamp"]).copy()
    df["user_id"] = df["user_id"].astype(str)
    df["item_id"] = df["item_id"].astype(str)
    df["timestamp"] = df["timestamp"].astype("int64")
    df = df.sort_values("timestamp").reset_index(drop=True)

    # приведём к единому виду как в твоём примере: from=user, to=item
    out = df[["user_id", "item_id", "timestamp"]].rename(columns={"user_id": "from", "item_id": "to"})
    return out


df_raw = load_ml100k_as_events(cfg.ml100k_path, cfg.sep)
df_raw.head(), df_raw.shape

(    from     to  timestamp
 0  u_259  i_255  874724710
 1  u_259  i_286  874724727
 2  u_259  i_298  874724754
 3  u_259  i_185  874724781
 4  u_259  i_173  874724843,
 (100000, 3))

Bipartite mapping + time starts at 0

In [3]:
def build_bipartite_id_maps(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, int], Dict[str, int]]:
    user_map: Dict[str, int] = {}
    item_map: Dict[str, int] = {}

    def map_user(x: str) -> int:
        if x not in user_map:
            user_map[x] = len(user_map)
        return user_map[x]

    def map_item(x: str) -> int:
        if x not in item_map:
            item_map[x] = len(item_map)
        return item_map[x]

    df = df.copy()
    df["from"] = df["from"].astype(str).map(map_user).astype("int64")
    df["to"] = df["to"].astype(str).map(map_item).astype("int64")

    item_offset = int(len(user_map))
    df["to"] = (df["to"] + item_offset).astype("int64")
    return df, user_map, item_map


df = df_raw.sort_values("timestamp").reset_index(drop=True)
t0 = int(df["timestamp"].min())
df["timestamp"] = (df["timestamp"] - t0).astype("int64")

df, user_map, item_map = build_bipartite_id_maps(df)

num_users = int(len(user_map))
num_items = int(len(item_map))
item_offset = int(num_users)
num_nodes = int(num_users + num_items)

df.head(), (num_users, num_items, num_nodes)

(   from   to  timestamp
 0     0  943          0
 1     0  944         17
 2     0  945         44
 3     0  946         71
 4     0  947        133,
 (943, 1682, 2625))

Temporal split 70-15-15 по событиям

In [4]:
def bounds_event_ratio_split(
    df: pd.DataFrame,
    train_ratio: float,
    val_ratio: float,
    time_col: str = "timestamp",
) -> Tuple[int, int]:
    df_sorted = df.sort_values(time_col).reset_index(drop=True)
    times = df_sorted[time_col].to_numpy()
    n = int(times.shape[0])

    idx_val = int(n * float(train_ratio))
    idx_test = int(n * float(train_ratio + val_ratio))

    val_time = int(times[idx_val])
    test_time = int(times[idx_test])
    return val_time, test_time


val_time, test_time = bounds_event_ratio_split(df, cfg.train_ratio, cfg.val_ratio, time_col="timestamp")
val_time, test_time

(12314561, 16311713)

Discretize timestamps → sid, подготовить события по split и snapshots (undirected

In [5]:
def gran_to_seconds(gran: str) -> int:
    g = gran.lower()
    if g == "h":
        return 3600
    if g == "d":
        return 86400
    raise ValueError("snapshot_gran must be 'h' or 'd'")


bin_sec = gran_to_seconds(cfg.snapshot_gran)

df = df.copy()
df["sid"] = (df["timestamp"] // bin_sec).astype("int64")

# split label by raw timestamp
df["split"] = np.where(
    df["timestamp"] < val_time, "train",
    np.where(df["timestamp"] < test_time, "val", "test")
)

# events (directed): from=user -> to=item
df_events = df[["from", "to", "timestamp", "sid", "split"]].copy()

# snapshots (undirected) for message passing: add reverse edges
df_rev = df_events.copy()
df_rev[["from", "to"]] = df_rev[["to", "from"]]
df_mp = pd.concat([df_events, df_rev], ignore_index=True)
df_mp = df_mp.drop_duplicates(subset=["from", "to", "timestamp"]).sort_values(["sid", "timestamp"]).reset_index(drop=True)

max_sid = int(df["sid"].max())
max_sid

214

Группировка по sid (events_by_sid + mp_by_sid)

In [6]:
def group_by_sid(df_in: pd.DataFrame, sid_col: str = "sid") -> Dict[int, pd.DataFrame]:
    out: Dict[int, pd.DataFrame] = {}
    for sid, g in df_in.groupby(sid_col, sort=True):
        out[int(sid)] = g.reset_index(drop=True)
    return out


mp_by_sid = group_by_sid(df_mp, "sid")

train_events_by_sid = group_by_sid(df_events[df_events["split"] == "train"], "sid")
val_events_by_sid = group_by_sid(df_events[df_events["split"] == "val"], "sid")
test_events_by_sid = group_by_sid(df_events[df_events["split"] == "test"], "sid")

(len(train_events_by_sid), len(val_events_by_sid), len(test_events_by_sid))

(142, 47, 26)

In [7]:
def select_last_event_per_user(batch_df: pd.DataFrame, item_offset: int) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Внутри одного sid берём ровно 1 позитив на пользователя: последний по timestamp.
    Возвращаем:
      users_global: [U]
      pos_items_local: [U]  (0..num_items-1)
    """
    if len(batch_df) == 0:
        return (
            torch.empty(0, dtype=torch.long),
            torch.empty(0, dtype=torch.long),
        )

    # сортируем (user, timestamp) и берём последний
    g = batch_df.sort_values(["from", "timestamp"], ascending=[True, True])
    # idx последнего события на каждого user
    last_idx = g.groupby("from", sort=False).tail(1).index
    picked = g.loc[last_idx]

    users_global = torch.tensor(picked["from"].to_numpy(dtype=np.int64), dtype=torch.long)
    pos_items_local = torch.tensor((picked["to"].to_numpy(dtype=np.int64) - item_offset), dtype=torch.long)
    return users_global, pos_items_local

Простая GCN-подобная нормализация + Encoder + DotProduct decoder

In [8]:
def build_norm_adj(edge_src: torch.Tensor, edge_dst: torch.Tensor, num_nodes: int, device: torch.device) -> torch.Tensor:
    """
    Возвращает sparse COO adjacency с GCN-нормализацией D^{-1/2} A D^{-1/2}.
    Добавляем self-loops.
    """
    # self loops
    self_idx = torch.arange(num_nodes, dtype=torch.long, device=device)
    src = torch.cat([edge_src, self_idx], dim=0)
    dst = torch.cat([edge_dst, self_idx], dim=0)

    # degree (по исходящим; для симметричного графа норм ок)
    deg = torch.bincount(src, minlength=num_nodes).float()
    deg_inv_sqrt = torch.pow(deg.clamp(min=1.0), -0.5)

    vals = deg_inv_sqrt[src] * deg_inv_sqrt[dst]  # [E]
    idx = torch.stack([src, dst], dim=0)          # [2, E]

    A = torch.sparse_coo_tensor(idx, vals, size=(num_nodes, num_nodes), device=device)
    A = A.coalesce()
    return A


class SimpleGCNEncoder(nn.Module):
    def __init__(self, in_dim: int, hid_dim: int, out_dim: int, n_layers: int, dropout: float):
        super().__init__()
        assert n_layers >= 1
        self.dropout = float(dropout)
        self.lins = nn.ModuleList()
        self.bns = nn.ModuleList()

        if n_layers == 1:
            self.lins.append(nn.Linear(in_dim, out_dim))
        else:
            self.lins.append(nn.Linear(in_dim, hid_dim))
            self.bns.append(nn.BatchNorm1d(hid_dim))
            for _ in range(n_layers - 2):
                self.lins.append(nn.Linear(hid_dim, hid_dim))
                self.bns.append(nn.BatchNorm1d(hid_dim))
            self.lins.append(nn.Linear(hid_dim, out_dim))

    def forward(self, A_norm: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
        # x: [N, in_dim]
        h = x
        if len(self.lins) == 1:
            h = torch.sparse.mm(A_norm, self.lins[0](h))
            return h

        for li in range(len(self.lins) - 1):
            h = torch.sparse.mm(A_norm, self.lins[li](h))
            h = self.bns[li](h)
            h = F.relu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)

        h = torch.sparse.mm(A_norm, self.lins[-1](h))
        return h


class DotProductDecoder(nn.Module):
    def forward(self, z_users: torch.Tensor, z_items: torch.Tensor) -> torch.Tensor:
        # [U, d] x [I, d]^T -> [U, I]
        return z_users @ z_items.t()

Model init (node_emb + encoder + decoder) + optimizer

In [9]:
node_emb = nn.Embedding(num_nodes, cfg.node_dim).to(device)
nn.init.normal_(node_emb.weight, std=0.1)

encoder = SimpleGCNEncoder(
    in_dim=cfg.node_dim,
    hid_dim=cfg.embed_dim,
    out_dim=cfg.embed_dim,
    n_layers=cfg.n_layers,
    dropout=cfg.dropout,
).to(device)

decoder = DotProductDecoder().to(device)

opt = torch.optim.Adam(
    list(node_emb.parameters()) + list(encoder.parameters()) + list(decoder.parameters()),
    lr=cfg.lr,
    weight_decay=cfg.weight_decay,
)

Prefix-window сборка (по sid) + forward для одного sid

In [10]:
from collections import deque
from typing import Deque

def concat_edges(edge_list: List[Tuple[torch.Tensor, torch.Tensor]]) -> Tuple[torch.Tensor, torch.Tensor]:
    if len(edge_list) == 0:
        return torch.empty(0, dtype=torch.long), torch.empty(0, dtype=torch.long)
    src = torch.cat([s for s, _ in edge_list], dim=0)
    dst = torch.cat([d for _, d in edge_list], dim=0)
    return src, dst


def mp_edges_for_sid(sid: int, mp_by_sid: Dict[int, pd.DataFrame], device: torch.device) -> Tuple[torch.Tensor, torch.Tensor]:
    g = mp_by_sid.get(int(sid))
    if g is None or len(g) == 0:
        return torch.empty(0, dtype=torch.long, device=device), torch.empty(0, dtype=torch.long, device=device)
    src = torch.tensor(g["from"].to_numpy(np.int64), dtype=torch.long, device=device)
    dst = torch.tensor(g["to"].to_numpy(np.int64), dtype=torch.long, device=device)
    return src, dst


def compute_z_from_prefix(window: Deque[Tuple[torch.Tensor, torch.Tensor]], num_nodes: int) -> torch.Tensor:
    src, dst = concat_edges(list(window))
    A = build_norm_adj(src, dst, num_nodes=num_nodes, device=device)
    z = encoder(A, node_emb.weight)
    return z

Train epoch: streaming по sid (без лика внутри sid)

In [11]:
def train_epoch_streaming(
    train_events_by_sid: Dict[int, pd.DataFrame],
    mp_by_sid: Dict[int, pd.DataFrame],
    max_sid: int,
    window_sids: int,
) -> float:
    encoder.train()
    node_emb.train()
    decoder.train()

    item_ids_global = torch.arange(item_offset, item_offset + num_items, device=device, dtype=torch.long)

    # sliding window по sid (0 => весь префикс)
    W = int(window_sids)
    window: Deque[Tuple[torch.Tensor, torch.Tensor]] = deque(maxlen=(W if W > 0 else None))

    # prefix pointer: какой sid уже добавили в window
    prefix_sid = 0

    total_loss = 0.0
    total_users = 0

    for sid in sorted(train_events_by_sid.keys()):
        sid = int(sid)

        # ДО sid добавляем snapshots в prefix
        while prefix_sid < sid:
            s, d = mp_edges_for_sid(prefix_sid, mp_by_sid, device=device)
            if s.numel() > 0:
                window.append((s, d))
            prefix_sid += 1

        # считаем z только по префиксу (< sid)
        z = compute_z_from_prefix(window, num_nodes=num_nodes)

        # текущие события sid: выбрать по одному позитиву на пользователя
        users_global_cpu, pos_items_local_cpu = select_last_event_per_user(train_events_by_sid[sid], item_offset)
        if users_global_cpu.numel() == 0:
            continue

        users_global = users_global_cpu.to(device)
        pos_items_local = pos_items_local_cpu.to(device)

        users_z = z[users_global]            # [U, d]
        items_z = z[item_ids_global]         # [I, d]
        logits = decoder(users_z, items_z)   # [U, I]

        loss = F.cross_entropy(logits, pos_items_local)

        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()

        U = int(users_global.numel())
        total_loss += float(loss.item()) * U
        total_users += U

    return float(total_loss / max(total_users, 1))

Eval: streaming по sid

In [12]:
@torch.no_grad()
def eval_streaming(
    events_by_sid: Dict[int, pd.DataFrame],
    mp_by_sid: Dict[int, pd.DataFrame],
    start_prefix_sid: int,
    window_sids: int,
    k: int,
) -> Dict[str, float]:
    encoder.eval()
    node_emb.eval()
    decoder.eval()

    item_ids_global = torch.arange(item_offset, item_offset + num_items, device=device, dtype=torch.long)
    k = int(k)

    W = int(window_sids)
    window: Deque[Tuple[torch.Tensor, torch.Tensor]] = deque(maxlen=(W if W > 0 else None))

    prefix_sid = int(start_prefix_sid)

    ndcg_by_user: Dict[int, List[float]] = {}
    hr_by_user: Dict[int, List[float]] = {}
    mrr_by_user: Dict[int, List[float]] = {}
    topk_union: set = set()

    for sid in sorted(events_by_sid.keys()):
        sid = int(sid)

        while prefix_sid < sid:
            s, d = mp_edges_for_sid(prefix_sid, mp_by_sid, device=device)
            if s.numel() > 0:
                window.append((s, d))
            prefix_sid += 1

        z = compute_z_from_prefix(window, num_nodes=num_nodes)

        users_global_cpu, pos_items_local_cpu = select_last_event_per_user(events_by_sid[sid], item_offset)
        if users_global_cpu.numel() == 0:
            continue

        users_global = users_global_cpu.to(device)
        pos_items_local = pos_items_local_cpu.to(device)

        users_z = z[users_global]
        items_z = z[item_ids_global]
        logits = decoder(users_z, items_z)  # [U, I]

        topk = torch.topk(logits, k=k, dim=1).indices  # [U, k] (local item ids)

        for row in range(int(users_global.numel())):
            u = int(users_global[row].item())
            pos = int(pos_items_local[row].item())

            hits = (topk[row] == pos).nonzero(as_tuple=False)
            if hits.numel() == 0:
                ndcg = 0.0
                hr = 0.0
                mrr = 0.0
            else:
                rank0 = int(hits[0].item())  # 0-based позиция в topk
                ndcg = 1.0 / float(np.log2(rank0 + 2.0))  # = 1/log2(rank+1) где rank=rank0+1
                hr = 1.0
                mrr = 1.0 / float(rank0 + 1.0)

            ndcg_by_user.setdefault(u, []).append(ndcg)
            hr_by_user.setdefault(u, []).append(hr)
            mrr_by_user.setdefault(u, []).append(mrr)

            # coverage union
            topk_union.update([int(x) for x in topk[row].detach().cpu().tolist()])

    # сначала по пользователю, потом по пользователям
    def mean_of_user_means(d: Dict[int, List[float]]) -> float:
        if len(d) == 0:
            return 0.0
        return float(np.mean([float(np.mean(v)) for v in d.values()]))

    ndcg = mean_of_user_means(ndcg_by_user)
    hr = mean_of_user_means(hr_by_user)
    mrr = mean_of_user_means(mrr_by_user)
    coverage = float(len(topk_union)) / float(max(1, num_items))

    return {"NDCG": ndcg, "HR": hr, "MRR": mrr, "Coverage": coverage}

Training runner

In [13]:
best_val = -1.0
best_state = None

# train prefix starts from sid=0
train_prefix_start = 0

# для val: prefix включает весь train-диапазон (< first val sid) автоматически через mp_by_sid + start_prefix_sid=0
# но удобнее стартовать с 0 и просто стримить val - prefix догонит sid сам
val_prefix_start = 0

for epoch in range(1, cfg.epochs + 1):
    loss = train_epoch_streaming(
        train_events_by_sid=train_events_by_sid,
        mp_by_sid=mp_by_sid,
        max_sid=max_sid,
        window_sids=cfg.window_sids,
    )

    val = eval_streaming(
        events_by_sid=val_events_by_sid,
        mp_by_sid=mp_by_sid,
        start_prefix_sid=val_prefix_start,
        window_sids=cfg.window_sids,
        k=cfg.k,
    )

    print(
        f"Epoch {epoch:03d} | loss={loss:.4f} | "
        f"val: NDCG@{cfg.k}={val['NDCG']:.4f} Cov@{cfg.k}={val['Coverage']:.4f}"
    )

    if val["NDCG"] > best_val:
        best_val = float(val["NDCG"])
        best_state = {
            "node_emb": {k: v.detach().cpu().clone() for k, v in node_emb.state_dict().items()},
            "encoder":  {k: v.detach().cpu().clone() for k, v in encoder.state_dict().items()},
            "decoder":  {k: v.detach().cpu().clone() for k, v in decoder.state_dict().items()},
            "epoch": epoch,
        }

print("Best val NDCG:", best_val, "at epoch", (best_state["epoch"] if best_state else None))

Epoch 001 | loss=8.4388 | val: NDCG@20=0.0176 Cov@20=0.0981
Epoch 002 | loss=7.1072 | val: NDCG@20=0.0234 Cov@20=0.1249
Epoch 003 | loss=6.9106 | val: NDCG@20=0.0219 Cov@20=0.1296
Epoch 004 | loss=6.7359 | val: NDCG@20=0.0287 Cov@20=0.1373
Epoch 005 | loss=6.5633 | val: NDCG@20=0.0288 Cov@20=0.1243
Epoch 006 | loss=6.4119 | val: NDCG@20=0.0401 Cov@20=0.1260
Epoch 007 | loss=6.2997 | val: NDCG@20=0.0297 Cov@20=0.1677
Epoch 008 | loss=6.1903 | val: NDCG@20=0.0331 Cov@20=0.1873
Epoch 009 | loss=6.0545 | val: NDCG@20=0.0310 Cov@20=0.1694
Epoch 010 | loss=5.9163 | val: NDCG@20=0.0328 Cov@20=0.1837
Epoch 011 | loss=5.9377 | val: NDCG@20=0.0296 Cov@20=0.1992
Epoch 012 | loss=5.7503 | val: NDCG@20=0.0182 Cov@20=0.2152
Epoch 013 | loss=5.6152 | val: NDCG@20=0.0255 Cov@20=0.2289
Epoch 014 | loss=5.5175 | val: NDCG@20=0.0208 Cov@20=0.2235
Epoch 015 | loss=5.4768 | val: NDCG@20=0.0250 Cov@20=0.2087
Epoch 016 | loss=5.3621 | val: NDCG@20=0.0262 Cov@20=0.1980
Epoch 017 | loss=5.1674 | val: NDCG@20=0

Restore best + Test eval

In [14]:
if best_state is not None:
    node_emb.load_state_dict(best_state["node_emb"])
    encoder.load_state_dict(best_state["encoder"])
    decoder.load_state_dict(best_state["decoder"])

test = eval_streaming(
    events_by_sid=test_events_by_sid,
    mp_by_sid=mp_by_sid,
    start_prefix_sid=0,              # prefix сам нарастёт до нужных sid
    window_sids=cfg.window_sids,
    k=cfg.k,
)
print(f"test: NDCG@{cfg.k}={test['NDCG']:.4f} Cov@{cfg.k}={test['Coverage']:.4f}")

test: NDCG@20=0.0207 Cov@20=0.1344
